# Notebook 04 — Evaluation and Timeline Generation
**Task 1:** Anomaly detection: threshold comparison, ROC curve, anomaly score plot.
**Task 2:** Timeline generation: sliding window, Gantt chart, confusion matrix, F1.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import json
import matplotlib.pyplot as plt
import torch

from src.config import MODELS_DIR, PLOTS_DIR, RESULTS_DIR, SEED, CLASSES, CLASS_TO_IDX, IDX_TO_CLASS
from src.model import Encoder, Decoder, LSTMHead, JointModel, freeze_decoder
from src.dataset import load_npy_split
from src.evaluate import (
    compute_reconstruction_errors, compute_threshold, find_optimal_threshold,
    threshold_comparison, evaluate_classifier, get_embeddings,
    plot_roc_curve, plot_anomaly_scores, plot_confusion_matrix,
    plot_per_class_metrics, plot_latent_space, plot_mse_distribution
)
from src.timeline import (
    build_test_sequence, sliding_window_predict,
    majority_vote_smooth, merge_segments,
    plot_timeline, plot_score_over_time, print_timeline
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)

## 1. Load Model and Data

In [ ]:
encoder = Encoder(); decoder = Decoder(); lstm_head = LSTMHead()
freeze_decoder(decoder)
joint_model = JointModel(encoder, decoder, lstm_head)
joint_model.load_state_dict(
    torch.load(MODELS_DIR / "joint_model_best.pth", map_location=DEVICE))
joint_model = joint_model.to(DEVICE).eval()
print("Joint model loaded.")

X_test, y_test = load_npy_split("test")
X_val,  y_val  = load_npy_split("val")
X_val_normal   = X_val[y_val == CLASS_TO_IDX["Normal"]]

print(f"Test clips : {X_test.shape}")
print(f"Label dist : {dict(zip(CLASSES, np.bincount(y_test)))}")

with open(RESULTS_DIR / "thresholds.json") as f:
    thresholds = json.load(f)
thr_2std = thresholds["mean_2std"]
thr_3std = thresholds["mean_3std"]
print(f"Thresholds: 2std={thr_2std:.5f}  3std={thr_3std:.5f}")

## 2. Task 1: Anomaly Detection

### 2a. Threshold Comparison

In [ ]:
results, normal_errors, test_errors, binary_gt = threshold_comparison(
    encoder=None, decoder=None,
    X_val_normal=X_val_normal,
    X_test=X_test, y_test=y_test,
    autoencoder=joint_model, device=DEVICE
)
thr_optimal = find_optimal_threshold(test_errors, binary_gt)
print(f"ROC-optimal threshold: {thr_optimal:.5f}")

### 2b. ROC Curve

In [ ]:
plot_roc_curve(test_errors, binary_gt, save_path=PLOTS_DIR / "roc_curve.png")
plt.show()

### 2c. Anomaly Score Plot

In [ ]:
plot_anomaly_scores(test_errors, y_test, threshold=thr_optimal,
                    save_path=PLOTS_DIR / "anomaly_scores.png")
plt.show()

### 2d. MSE Distribution

In [ ]:
X_test_normal = X_test[y_test == CLASS_TO_IDX["Normal"]]
X_test_anom   = X_test[y_test != CLASS_TO_IDX["Normal"]]
plot_mse_distribution(joint_model, X_test_normal, X_test_anom,
    save_path=PLOTS_DIR / "mse_distribution_test.png", device=DEVICE)
plt.show()

## 3. Task 2: Event Classification

### 3a. Per-class Metrics

In [ ]:
y_pred, report = evaluate_classifier(joint_model, X_test, y_test, device=DEVICE)
plot_per_class_metrics(report, save_path=PLOTS_DIR / "per_class_metrics.png")
plt.show()

### 3b. Confusion Matrix

In [ ]:
plot_confusion_matrix(y_test, y_pred, save_path=PLOTS_DIR / "confusion_matrix.png")
plt.show()

## 4. Latent Space Visualisation

In [ ]:
embeddings = get_embeddings(encoder.to(DEVICE), X_test, device=DEVICE)
print(f"Embeddings shape: {embeddings.shape}")

plot_latent_space(embeddings, y_test, method="pca",
    save_path=PLOTS_DIR / "pca_latent.png")
plt.show()

plot_latent_space(embeddings, y_test, method="tsne",
    save_path=PLOTS_DIR / "tsne_latent.png")
plt.show()

## 5. Timeline Generation

### 5a. Build Synthetic Surveillance Session

In [ ]:
seq_clips, seq_labels, segment_info = build_test_sequence(X_test, y_test, clips_per_segment=20)
print(f"Sequence: {len(seq_clips)} clips")
for cls, s, e in segment_info:
    print(f"  [{cls}] clips {s}-{e}")

### 5b. Sliding Window Inference

In [ ]:
pred_labels, pred_probs, recon_errors = sliding_window_predict(
    joint_model, seq_clips, device=DEVICE)
print(f"Predictions: {len(pred_labels)} clips")

### 5c. Smooth and Merge

In [ ]:
smoothed = majority_vote_smooth(pred_labels, window=3)
segments = merge_segments(smoothed, recon_errors, threshold=thr_optimal, fps=1.0)
print_timeline(segments)

### 5d. Score Over Time

In [ ]:
plot_score_over_time(recon_errors, pred_labels, threshold=thr_optimal,
    save_path=PLOTS_DIR / "score_over_time.png")
plt.show()

### 5e. Gantt Timeline Chart

In [ ]:
plot_timeline(segments, save_path=PLOTS_DIR / "timeline.png")
plt.show()

## 6. Save All Results

In [ ]:
final_results = {
    "threshold_comparison": results,
    "classification_report": report,
    "timeline_segments": segments,
}
with open(RESULTS_DIR / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2, default=float)

print(f"[DONE] Results saved to {RESULTS_DIR}")